In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Crear widgets
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_cr")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsfinalproject")

In [0]:
# Obtener valores de widgets
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

# Obtener ruta de crime_reports.csv
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/crime-reports.csv"

In [0]:
# Crear DataFrame df_crime_reports infiriendo el schema.
df_crime_reports = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
# Definir esquema para el DataFrame df_crime_reports
# emergency_district se lee como IntegerType (0/1) y luego se convierte a Boolean
crime_reports_schema = StructType(fields=[StructField("report_year", IntegerType(), True),
                                     StructField("month_id", IntegerType(), True),
                                     StructField("department", StringType(), True),
                                     StructField("province", StringType(), True),
                                     StructField("district", StringType(), True),
                                     StructField("ubigeo", StringType(), True),
                                     StructField("crime_id", IntegerType(), True),
                                     StructField("n_crimes", IntegerType(), True),
                                     StructField("emergency_district", IntegerType(), True)
])

In [0]:
# Crear Dataframe df_crime_reports_final con el esquema anterior.
# Convertir emergency_district de 0/1 a Boolean (1=True, 0=False)
df_crime_reports_final = spark.read\
.option('header', True)\
.schema(crime_reports_schema)\
.csv(ruta)\
.withColumn("emergency_district", col("emergency_district") == 1)

In [0]:
# Añadir columna report_id a DataFrame (comenzando desde 1)
df_crime_reports_final = df_crime_reports_final.withColumn("report_id", monotonically_increasing_id() + 1)

In [0]:
# Seleccionar columnas específicas, con report_id al inicio.
df_crime_reports_reordered = df_crime_reports_final.select(
    col("report_id"),
    col("report_year"),
    col("month_id"),
    col("department"),
    col("province"),
    col("district"),
    col("ubigeo"),
    col("crime_id"),
    col("n_crimes"),
    col("emergency_district")
)

In [0]:
# Añadir columna ingestion_date a DataFrame.
df_crime_reports_final = df_crime_reports_reordered.withColumn("ingestion_date", current_timestamp())

In [0]:
# Ingestar data en la tabla crime_reports
df_crime_reports_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.crime_reports")